# subprocess 

subprocess 是 Python 的标准库，它是你与操作系统交互的“瑞士军刀”。
- 简单来说，它允许你在 Python 程序中启动新的进程、执行系统命令（如 ls, git, docker）、调用外部脚本，并控制它们的输入输出。

- 它的核心设计哲学是“显式优于隐式”和“安全优先”，旨在替代老旧且不安全的 os.system()、os.popen() 等模块。

以下是 subprocess 的深度解析，涵盖核心用法、最佳实践和安全避坑指南。

## 🚀 核心接口：subprocess.run() (推荐)
- 这是 Python 3.5+ 引入的“高级接口”，也是官方推荐的首选方法。它封装了底层的复杂性，采用“执行—等待—返回结果”的阻塞模式，非常适合执行一次性命令。

### 基本语法

In [ ]:
import subprocess

# 执行命令，捕获输出
result = subprocess.run(
    ["ls", "-l"],          # 命令和参数（列表形式）
    capture_output=True,   # 捕获标准输出和错误
    text=True,             # 以字符串形式返回（而非字节）
    check=True             # 如果返回码非0，自动抛出异常
)

print("输出:\n", result.stdout)

| 参数 | 作用 | 最佳实践 |
| :--- | :--- | :--- |
| `args` | 要执行的命令。 | 始终使用列表（如 `["git", "commit"]`），避免 Shell 注入风险。 |
| `capture_output` | 设为 `True` 可捕获 stdout 和 stderr。 | Python 3.7+ 推荐用法，替代繁琐的 `stdout=PIPE`。 |
| `text` / `universal_newlines` | 设为 `True` 以文本模式（字符串）处理 I/O。 | 默认是二进制（bytes），处理文本务必开启此选项。 |
| `check` | 设为 `True`，若命令执行失败（返回码!=0）则抛出 `CalledProcessError`。 | 建议开启，防止错误被忽略。 |
| `timeout` | 设置超时时间（秒）。 | 防止子进程卡死拖垮主程序，超时会自动杀掉子进程。 |
| `shell` | 是否通过 Shell 执行。 | 默认 False。除非必须使用管道符 `|` 等 Shell 特性，否则严禁设为 True。 |



## 🛠️ 底层核心：subprocess.Popen
- Popen 是 subprocess 模块的基石，提供了对进程生命周期的完全控制权。如果你需要非阻塞执行、实时读取日志流或与进程进行交互式通信，必须使用它。

### 典型场景：实时读取输出

In [ ]:
import subprocess

# 启动进程（不等待结束）
process = subprocess.Popen(
    ["ping", "127.0.0.1", "-n", "5"], # Windows下ping 5次
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# 实时逐行读取输出
for line in process.stdout:
    print(f"实时日志: {line.strip()}")

# 等待进程结束并获取返回码
process.wait()
print(f"进程结束，返回码: {process.returncode}")

### Popen 与 run 的关系
- <font color='red'>subprocess.run() 本质上是对 Popen 的一层封装。run() 内部调用了 Popen，并自动执行了 communicate() 来等待进程结束。</font>

## ⚠️ 安全红线：Shell 注入风险
- 这是使用 subprocess 最需要警惕的地方。
### ❌ 危险做法
- 当 shell=True 时，Python 会将字符串直接交给系统 Shell (/bin/sh 或 cmd.exe) 解析，攻击者可以通过拼接恶意命令（如 ; rm -rf /）进行注入。

In [ ]:
# 极度危险！如果 filename 包含 "; rm -rf /"，整个系统可能被清空
filename = input("输入文件名: ")
subprocess.run(f"cat {filename}", shell=True) 

### ✅ 安全做法
- 原则：始终使用列表传参，并保持 shell=False（默认值）

In [ ]:
# 安全！参数作为列表传递，Python 会直接调用 exec 系统调用，绕过 Shell 解析
filename = input("输入文件名: ")
subprocess.run(["cat", filename]) 

## 🔄 进阶功能
1. 管道通信
- 你可以将一个进程的输出作为另一个进程的输入（类似 Linux 的 | 管道）。

In [ ]:
# 实现 "cat file.txt | grep 'error'"
p1 = subprocess.Popen(["cat", "app.log"], stdout=subprocess.PIPE)
p2 = subprocess.Popen(["grep", "error"], stdin=p1.stdout, stdout=subprocess.PIPE)
p1.stdout.close()  # 允许 p1 接收 SIGPIPE 信号

output, _ = p2.communicate()
print(output.decode())

2. 输入数据
- 你可以直接向子进程的标准输入发送数据

In [ ]:
# 向 python 解释器发送代码
process = subprocess.Popen(
    ["python"], 
    stdin=subprocess.PIPE, 
    stdout=subprocess.PIPE, 
    text=True
)
# 发送 print("Hello") 并换行
output, _ = process.communicate('print("Hello from subprocess!")\n')
print(output)

3. 环境变量与工作目录
- cwd: 指定子进程的工作目录（例如在指定目录下执行 git pull）。
- env: 定制子进程的环境变量（注意：默认会继承父进程所有环境变量，可能泄露密钥，建议显式构造最小化环境）。

## 📌 总结与最佳实践
1. 首选 subprocess.run()：对于大多数简单的命令执行，它是最简洁、最安全的。
2. 拒绝 shell=True：除非你非常清楚自己在做什么，否则永远不要开启 Shell 模式，始终使用列表传参。
3. 处理超时：始终设置 timeout 参数，防止外部程序挂起导致你的程序卡死。
4. 异常处理：使用 check=True 或在 try...except subprocess.CalledProcessError 块中执行，确保能捕获命令执行失败的错误。
5. 跨平台兼容：注意 Windows 和 Linux 的命令差异（如 dir vs ls），尽量使用 Python 标准库（如 os, shutil）替代简单的系统命令，只在必须调用外部工具时使用 subprocess。

# subprocess 模块的方法可以分为两类：
- 一类是 Python 3.5 之后推荐的高级接口（如 run），<font color='red'>它们是对底层逻辑的封装，简单易用；</font>
- 另一类是底层接口（如 Popen），提供了对进程的完全控制权。

以下是 subprocess 模块中所有主要方法的详细介绍和对比。

## 🚀 核心方法：subprocess.run()
这是 Python 3.5+ 引入的首选方法，也是官方最推荐的高级接口。它的设计哲学是“一站式”解决：执行命令、等待完成、返回结果。
- 功能：执行命令，等待其终止，并返回一个 CompletedProcess 对象，其中包含返回码、标准输出和标准错误等信息。
- 适用场景：绝大多数需要执行外部命令的场景，特别是当你需要获取命令的输出结果或检查其是否成功时。
- 关键参数：
    - args: 命令及其参数，推荐使用列表形式，如 ["ls", "-l"]。
    - capture_output: 设为 True 以捕获标准输出和标准错误（Python 3.7+）。
    - text (或 universal_newlines): 设为 True 以字符串（而非字节）形式处理输入输出。
    - check: 设为 True，如果命令返回非零状态码，则抛出 CalledProcessError 异常。
    - timeout: 设置超时时间（秒），防止子进程卡死。

In [ ]:
import subprocess

result = subprocess.run(
    ["echo", "Hello World"],
    capture_output=True,
    text=True,
    check=True
)
print(result.stdout) # 输出: Hello World

## ⚙️ 底层核心：subprocess.Popen

Popen 是 subprocess 模块的基石。所有其他高级函数（如 run, call 等）都是基于 Popen 封装实现的。它提供了对子进程生命周期的完全控制。
- 功能：创建并启动一个子进程，但不会等待其完成。它返回一个 Popen 对象，通过该对象可以与子进程进行交互。
- 适用场景：
    - 需要实时读取子进程的输出流（例如，实时显示日志）。
    - 需要与子进程进行双向通信（发送输入，读取输出）。
    - 需要非阻塞地启动一个长期运行的后台任务。
- 关键方法：
    - wait(): 等待子进程结束。
    - communicate(input, timeout): 与子进程交互，发送数据并读取输出，这是获取 Popen 输出最安全的方式。
    - poll(): 检查子进程是否已结束，如果未结束则返回 None。
    - terminate() / kill(): 终止或杀死子进程。

In [ ]:
import subprocess

# 启动一个长时间运行的进程
process = subprocess.Popen(["ping", "127.0.0.1", "-n", "5"], stdout=subprocess.PIPE, text=True)

# 实时读取输出
for line in process.stdout:
    print(f"实时日志: {line.strip()}")

# 等待进程结束
process.wait()
print(f"进程结束，返回码: {process.returncode}")

## 📜 遗留方法 (不推荐在新代码中使用)
在 subprocess.run() 出现之前，以下方法被广泛使用。它们现在被认为是“遗留”的，官方建议在新代码中优先使用 run()。
### subprocess.call()
- 功能：执行命令并等待其完成，然后仅返回命令的退出状态码（0 表示成功，非 0 表示失败）。
- 缺点：无法获取命令的输出内容，功能单一。
- 等价于：subprocess.run(...).returncode。
### subprocess.check_call()
- 功能：与 call() 类似，但如果命令返回非零状态码，则会抛出 CalledProcessError 异常。
- 缺点：同样无法获取命令的输出。
- 等价于：subprocess.run(..., check=True)。
### subprocess.check_output()
- 功能：执行命令，如果成功（返回码为 0）则返回命令的标准输出，否则抛出异常。
- 缺点：主要用于获取输出，错误处理不如 run() 灵活。
- 等价于：subprocess.run(..., capture_output=True, check=True).stdout。
### subprocess.getoutput() 和 subprocess.getstatusoutput()
- 功能：这两个是来自 Python 2.x 的遗留函数。getoutput() 返回命令的完整输出（包括标准错误），getstatusoutput() 返回一个包含（状态码, 输出）的元组。
- 缺点：它们隐式地调用系统 Shell (shell=True)，存在 Shell 注入的安全风险，并且行为在不同平台上可能不一致。应绝对避免在新代码中使用。

## 📊 方法对比与选择指南
为了让你更直观地选择合适的方法，可以参考下表：

| 方法 | 返回值 | 是否等待 | 能否捕获输出 | 异常处理 | 推荐度 |
| :--- | :--- | :--- | :--- | :--- | :--- |
| `run()` | `CompletedProcess` | 是 | 能 | 可选 (`check=True`) | ⭐⭐⭐⭐⭐ (首选) |
| `Popen()` | `Popen` 对象 | 否 | 能 (需手动) | 手动处理 | ⭐⭐⭐⭐ (高级控制) |
| `call()` | 状态码 (int) | 是 | 不能 | 不抛异常 | ⭐ (遗留) |
| `check_call()` | 状态码 (int) | 是 | 不能 | 抛异常 | ⭐ (遗留) |
| `check_output()` | 输出 (bytes/str) | 是 | 能 | 抛异常 | ⭐⭐ (特定场景) |

## 总结建议：
1. 默认使用 subprocess.run()：它功能最全，接口最清晰，能满足 90% 的需求。
2. 需要实时交互或后台任务时用 subprocess.Popen()：当你需要精细控制进程，比如实时读取日志流时，它是唯一选择。
3. 避免使用其他遗留方法：call, check_call, check_output 等功能都可以被 run() 优雅地替代。getoutput 等方法则因安全问题应避免使用。